# Tuần 2 — PhoBERT Finetune v2.6 (Focal Loss + LR Warmup)
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

**Thay đổi so với v2.4:**
- **Focal Loss** (γ=2) thay cross_entropy
  - `loss = -(1-p_true)^2 × log(p_true) × w_class`
  - Down-weight easy samples (absent đã đúng), focus vào hard/rare aspects
  - Tương thích hoàn toàn với class weights
- **LR warmup-peak-decay** (ds4v style)
  - `1.5e-5 → warmup → 3e-5 (peak) → cosine decay → 1.5e-6`
- **early_stop_patience = 3** (giảm từ 7)

> Chạy theo thứ tự từ Cell 1 đến Cell 9.

In [ ]:
# ============================================================
# Cell 1 — Check GPU & Install dependencies
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}')
    print(f'   VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('❌ Không có GPU! Kaggle: Settings → Accelerator → GPU T4 x2')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning {REPO_URL} (branch: {REPO_BRANCH})...')
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print('✅ Clone xong')
else:
    print('Repo đã tồn tại — pulling latest...')
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for d in ['data', 'outputs/models', 'outputs/results', 'outputs/eda']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, 'code/week1')
sys.path.insert(0, 'code/week2')
print('✅ Paths OK')

In [ ]:
# ============================================================
# Cell 3 — Download dataset VLSP 2018
# ============================================================
import pandas as pd, os

if not os.path.exists('data/train.csv'):
    print('Downloading VLSP 2018 Hotel dataset...')
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print('✅ Data downloaded')
else:
    print('✅ Data đã tồn tại')

for split in ['train', 'dev', 'test']:
    df = pd.read_csv(f'data/{split}.csv')
    print(f'  {split}: {len(df)} rows')

In [ ]:
# ============================================================
# Cell 4 — Preprocessing (VnCoreNLP)
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists('data/train_preprocessed.csv'):
    print('✅ Cache đã có (data/*_preprocessed.csv)')
    s = pd.read_csv('data/train_preprocessed.csv').iloc[0]
    print(f'  Original : {s["Review"][:80]}')
    print(f'  Processed: {str(s.get("processed_review", "N/A"))[:80]}')
else:
    print('Chưa có cache — chạy preprocessing với VnCoreNLP...')
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ['train', 'dev', 'test']:
        df = pd.read_csv(f'data/{split}.csv')
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f'data/{split}_preprocessed.csv')
        print(f'  ✅ {split}: {len(df)} rows')
    segmenter.close()
    print('✅ Preprocessing hoàn tất')

In [ ]:
# ============================================================
# Cell 5 — Verify config v2.6
# ============================================================
import json, os
from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

print('=== Train Config v2.6 ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')

# Assertions
assert TRAIN_CONFIG['learning_rate'] == 3e-5,    'peak LR phải là 3e-5'
assert TRAIN_CONFIG['lr_alpha'] == 0.1,          'alpha phải là 0.1'
assert TRAIN_CONFIG['early_stop_patience'] == 3, 'patience phải là 3'
assert TRAIN_CONFIG['encoder_option'] == 'cls_only'

# Verify focal loss trong model.py
src = open('code/week2/model.py').read()
assert 'focal_gamma' in src,         'Focal loss chưa được implement'
assert 'focal_factor' in src,        'Focal factor chưa có'
assert 'binary_cross_entropy' not in src, 'BCE cũ còn sót'

# Verify warmup scheduler trong train.py
train_src = open('code/week2/train.py').read()
assert 'lr_lambda' in train_src,     'LR lambda scheduler chưa có'
assert 'peak_lr' in train_src,       'peak_lr chưa có'

peak_lr = TRAIN_CONFIG['learning_rate']
initial_lr = peak_lr / 2
min_lr = TRAIN_CONFIG['lr_alpha'] * initial_lr
print(f'\n=== LR Schedule v2.6 ===')
print(f'  Initial: {initial_lr:.2e}')
print(f'  Peak:    {peak_lr:.2e}  (after {TRAIN_CONFIG["warmup_ratio"]*100:.0f}% warmup)')
print(f'  Min:     {min_lr:.2e}  (end of training)')
print(f'  Patience: {TRAIN_CONFIG["early_stop_patience"]} epochs')
print(f'\n✅ Config v2.6 OK — Focal Loss + LR warmup + patience=3')

In [ ]:
# ============================================================
# Cell 6 — TRAIN MAIN: cls_only (v2.6)
# Focal Loss + LR warmup-peak-decay + patience=3
# ============================================================
import torch
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics_cls = main(encoder_option='cls_only', use_amp=True)

print('\n' + '='*60)
print('MAIN RUN v2.6 — cls_only + Focal Loss + LR warmup')
print(f"  ACD F1:      {test_metrics_cls['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics_cls['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics_cls['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print('\nSo sánh các phiên bản:')
print('  Gốc  cls_only  (LR=1e-4, Adam, CE):           0.5543')
print('  v2.4 cls_only  (LR=3e-5, AdamW, CE):          0.5583')
print('  v2.5 cls_only  (LR=3e-5, AdamW, flat BCE): ❌ 0.4763')
print(f"  v2.6 cls_only  (Focal+warmup+patience=3):     {test_metrics_cls['macro_combined_f1']:.4f}")
print('='*60)

In [ ]:
# ============================================================
# Cell 7 — Ablation: concat_4_layers
# ============================================================
import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_concat = main(encoder_option='concat_4_layers', use_amp=True)

print('\n' + '='*55)
print('ABLATION v2.6 — concat_4_layers')
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
print(f"  concat_4_layers: {test_metrics_concat['macro_combined_f1']:.4f}")
gain = test_metrics_cls['macro_combined_f1'] - test_metrics_concat['macro_combined_f1']
print(f"  Diff: {gain*100:+.2f}%")
print('='*55)

In [ ]:
# ============================================================
# Cell 8 — Learning Curve
# ============================================================
import json, math, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open('outputs/results_cls_only/training_history.json'))
best_ep = history['best_epoch']
epochs  = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PhoBERT cls_only v2.6 (Focal+LR warmup) — Learning Curve', fontsize=13)

ax1.plot(epochs, history['train_loss'], 'o-', c='crimson',   lw=2, label='Train Loss')
ax1.plot(epochs, history['dev_loss'],   'o-', c='steelblue', lw=2, label='Dev Loss')
ax1.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (ep {best_ep})')
ax1.set(title='Loss', xlabel='Epoch', ylabel='Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['dev_acd_f1'],      's-', c='darkorange', lw=2,   label='Dev ACD F1')
ax2.plot(epochs, history['dev_spc_f1'],      '^-', c='purple',     lw=2,   label='Dev SPC F1')
ax2.plot(epochs, history['dev_combined_f1'], 'o-', c='green',      lw=2.5, label='Dev Combined F1')
ax2.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (ep {best_ep})')
ax2.axhline(0.7732, c='red',  ls=':', alpha=0.5, label='SOTA 0.7732')
ax2.axhline(0.5583, c='blue', ls=':', alpha=0.5, label='v2.4 0.5583')
ax2.set(title='F1 Score', xlabel='Epoch', ylabel='Macro F1'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/eda/learning_curve_v26.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'  Best epoch:       {best_ep} / {len(history["train_loss"])}')
print(f'  Best Combined F1: {history["best_combined_f1"]:.4f}')
print(f'  vs v2.4: {history["best_combined_f1"]-0.5583:+.4f}')
print(f'  vs gốc:  {history["best_combined_f1"]-0.5543:+.4f}')

In [ ]:
# ============================================================
# Cell 9 — Summary & Download
# ============================================================
import os, json, shutil

print('=== Số liệu v2.6 ===')
if os.path.exists('outputs/results_cls_only/week2_test_metrics.json'):
    m = json.load(open('outputs/results_cls_only/week2_test_metrics.json'))
    print(f"  cls_only ACD F1:      {m['macro_acd_f1']:.4f}")
    print(f"  cls_only SPC F1:      {m['macro_spc_f1']:.4f}")
    print(f"  cls_only Combined F1: {m['macro_combined_f1']:.4f}")
    print(f"\n  vs v2.4 (0.5583): {m['macro_combined_f1']-0.5583:+.4f}")
    print(f"  vs gốc  (0.5543): {m['macro_combined_f1']-0.5543:+.4f}")
    print(f"  vs SOTA (0.7732): {m['macro_combined_f1']-0.7732:+.4f}")

if os.path.exists('outputs/results/week2_test_metrics.json'):
    m2 = json.load(open('outputs/results/week2_test_metrics.json'))
    print(f"\n  concat_4_layers Combined F1: {m2['macro_combined_f1']:.4f} (ablation)")

shutil.make_archive('/kaggle/working/week2_v26_results', 'zip', 'outputs')
print('\n✅ Zip: /kaggle/working/week2_v26_results.zip → Kaggle Output panel → Download')